In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd
import chardet 

In [3]:
file_path = 'IRENA_RenewableEnergy_Statistics_2000-2022.csv'

with open(file_path, 'rb') as f:
    result = chardet.detect(f.read())

df_irena = pd.read_csv(file_path, encoding=result['encoding'])

file_path_1 = 'organised_Gen.csv'

with open(file_path_1, 'rb') as f:
    result = chardet.detect(f.read())

df_us_data = pd.read_csv(file_path_1, encoding=result['encoding'])

file_path_2 = '02 modern-renewable-energy-consumption.csv'

with open(file_path_2, 'rb') as f:
    result = chardet.detect(f.read())

df_world_data = pd.read_csv(file_path_2, encoding=result['encoding'])

In [7]:
# Drop rows with missing values and reset index
df_knn = df_us_data.dropna(subset=["GENERATION (Megawatthours)"]).copy()

# Convert categorical columns 
df_encoded = pd.get_dummies(df_knn, columns=["STATE", "TYPE OF PRODUCER"])

# Subset of dataset
df_sample = df_encoded.sample(n=10000, random_state=42)
y_sample = df_knn.loc[df_sample.index, "ENERGY SOURCE"]

X_sample = df_sample.drop(columns=["ENERGY SOURCE", "GENERATION (Megawatthours)", "Unnamed: 0"])
y_sample = y_sample.reset_index(drop=True)
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

# Encode labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_sample)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Evaluate KNN using different distance metrics
metrics = ["euclidean", "manhattan", "chebyshev"]
results = {}

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, output_dict=True)
    results[metric] = report

# Format results for display
report_df = pd.DataFrame({metric: {k: v['f1-score'] for k, v in results[metric].items() if isinstance(v, dict)} for metric in metrics})

In [9]:
results

{'euclidean': {'Coal': {'precision': 0.09018567639257294,
   'recall': 0.17258883248730963,
   'f1-score': 0.11846689895470383,
   'support': 197.0},
  'Geothermal': {'precision': 0.12244897959183673,
   'recall': 0.35294117647058826,
   'f1-score': 0.18181818181818182,
   'support': 17.0},
  'Hydroelectric Conventional': {'precision': 0.1206896551724138,
   'recall': 0.21705426356589147,
   'f1-score': 0.15512465373961218,
   'support': 129.0},
  'Natural Gas': {'precision': 0.19858156028368795,
   'recall': 0.18064516129032257,
   'f1-score': 0.1891891891891892,
   'support': 310.0},
  'Nuclear': {'precision': 0.06172839506172839,
   'recall': 0.08064516129032258,
   'f1-score': 0.06993006993006994,
   'support': 62.0},
  'Other': {'precision': 0.11538461538461539,
   'recall': 0.10344827586206896,
   'f1-score': 0.10909090909090909,
   'support': 145.0},
  'Other Biomass': {'precision': 0.14285714285714285,
   'recall': 0.12698412698412698,
   'f1-score': 0.13445378151260504,
   'su

In [8]:
report_df

,euclidean,manhattan,chebyshev
Coal,0.118467,0.125217,0.135364
Geothermal,0.181818,0.216216,0.126984
Hydroelectric Conventional,0.155125,0.149856,0.138889
Natural Gas,0.189189,0.210702,0.176871
Nuclear,0.069930,0.085714,0.068027
Other,0.109091,0.089552,0.092857
Other Biomass,0.134454,0.112360,0.127072
Other Gases,0.151515,0.122137,0.125984
Petroleum,0.140351,0.150110,0.162162
Pumped Storage,0.061538,0.059701,0.085714
